## CREATE A MODEL

In [2]:
import importlib.util
# Import pyflamegpu and some other libraries we will use in the tutorial
import pyflamegpu
import sys, random, math
import matplotlib.pyplot as plt

In [3]:
%env CUDA_PATH=D:\cuda12.9

env: CUDA_PATH=D:\cuda12.9


In [4]:
def create_model():
    model = pyflamegpu.ModelDescription("so-phy-shelter")
    return model

Agent 相关的东西

| Model Variable | Agent Variable Name | Description
| :--- | :--- | :--- |
| $AIE$ | `Around_in_evacuation` | agent周围在疏散的人 |
| $m$ | `metabolism` | Metabolism |
| $w$ | `sugar_level` | Sugar Wealth |
|  | `env_sugar_max` | Each cell has a maximum sugar level which can be no greater than $S_{max}$

In [6]:
student_agent=model.newAgent("student_agent")

student_agent.newstate("focused")
student_agent.newstate("not evacuate")
student_agent.newstate("building evacuate")
student_agent.newstate("stairwell evacuate")
student_agent.newstate("neighborhood evacuate")

student_agent.newVariableInt("x")
student_agent.newVariableInt("y")
student_agent.newVariableInt("agent_id")
student_agent.newVariableInt("target_stairwell_id")
student_agent.newVariableInt("target_shelter_id")



NameError: name 'model' is not defined

In [ ]:
def define_agents(model):
    """
        student agent
    """

    #create the agent
    agent = model.newAgent("student_agent")

    # Assign its variables
    agent.newVariableFloat("x")
    agent.newVariableFloat("y")
    agent.newVariableFloat("vx")
    agent.newVariableFloat("vy")
    agent.newVariableFloat("steer_x")
    agent.newVariableFloat("steer_y")
    agent.newVariableInt("agent_id")
    agent.newVariableInt("target_stairwell_id")
    agent.newVariableInt("target_shelter_id")

    # Assign its functions
    find_stairwell_fn = agent.newRTCFunction("find_stairwell", pyflamegpu.codegen.translate(find_stairwell))
    find_shelter_fn = agent.newRTCFunction("find_shelter", pyflamegpu.codegen.translate(find_shelter))
    walk_move_fn = agent.newRTCFunction("walk_move", pyflamegpu.codegen.translate(walk_move))
    downstair_move_fn = agent.newRTCFunction("downstair_move", pyflamegpu.codegen.translate(downstair_move))

    """
        stairwell agent
    """   

    #create the agent
    agent = model.newAgent("stairwell_agent")

## Define Environmental Properties

In [3]:
def define_environment(model):
    """
        Environment
    """
    env = model.Environment()

    #activate_radium
    env.newPropertyFloat("Range_be_activated", 15.0)
    env.newPropertyFloat("Range_find_stairwell", 100.0)
    env.newPropertyFloat("Range_find_shelter", 500.0)
    env.newPropertyFloat("threshold_stairwell_congestion", 10.0)
    
    

# Define Messages

In [ ]:
def define_messages(model):
    """
      Location messages
    """  
    message = model.newMessageBruteForce("student_agent_location_message")
    message.newVariableID("id")
    message.newVariableFloat("x")
    message.newVariableFloat("y")

    message = model.newMessageBruteForce("shelter_agent_location_message")
    message.newVariableID("id")
    message.newVariableFloat("x")
    message.newVariableFloat("y")

    message = model.newMessageBruteForce("stairwell_agent_location_message")
    message.newVariableID("id")
    message.newVariableFloat("x")
    message.newVariableFloat("y")


    """
      stairwell state messages
    """
    message = model.newMessageBruteForce("stairwell_congestion_message")
    message.newVariableID("id")
    message.newVariableFloat("congestion_state")

    """
      student agent state messages
    """
    message = model.newMessageBruteForce("student_evacuate_state_message")
    message.newVariableID("id")
    message.newVariableFloat("evacuate state")



## functions

## 初始化population

我现在集齐了碎片：population的地理位置，状态，速度，熟悉度的差异。

熟悉度可以参考设置不同的策略

也可以设置各种问卷：例如你们会不会倾向于用手机联系同学。

小trick：
参考博弈论的code部分
- 可以为evacuees 的疏散状态设置为特征值
- 可以用随机的方法随机一个，然后添加到状态里面
~~~python
energy = max(
    random.normalvariate(INIT_ENERGY_MU, INIT_ENERGY_SIGMA), INIT_ENERGY_MIN
)
if MAX_ENERGY > 0.0:
    energy = min(energy, MAX_ENERGY)
instance.setVariableFloat("energy", energy)
~~~
